# How to Train **YOLO26** Object Detection on a Custom Dataset
### (Drowsy Driver — 6 class) — theo đúng hướng dẫn Roboflow

YOLO26 dùng kiến trúc hợp nhất, **end-to-end (NMS-free)**, optimizer **MuSGD**.

Dataset: `close_eyeL, close_eyeR, no_yawn, open_eyeL, open_eyeR, yawn`

`Runtime → Change runtime type → T4 GPU → Run all`

## Setup
### Cấu hình API key
Notebook thử lấy key từ **Colab Secrets** (🔑 tên `ROBOFLOW_API_KEY`); nếu chưa set thì dùng key nhúng sẵn bên dưới.

In [ ]:
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    assert ROBOFLOW_API_KEY
    print('✅  Dùng API key từ Colab Secrets')
except Exception:
    ROBOFLOW_API_KEY = 'qI3lEKlNpIZpNENdk3MH'    # ← key của bạn
    print('✅  Dùng API key nhúng sẵn')

### Before you start — kiểm tra GPU

In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print(HOME)
MODEL = 'yolo26s'    # đổi: yolo26n (nhẹ) / yolo26m / yolo26l (mAP cao hơn). Guide dùng yolo26m.

### Install dependencies required for YOLO26

In [ ]:
%pip install -q "ultralytics>=8.4.0" supervision roboflow
# không cho ultralytics theo dõi hoạt động
!yolo settings sync=False
import ultralytics
ultralytics.checks()

## Inference với model pre-trained trên COCO
### Download ảnh ví dụ

In [ ]:
!wget -q https://media.roboflow.com/notebooks/examples/dog-2.jpeg
!wget -q https://media.roboflow.com/notebooks/examples/dog-3.jpeg

### CLI

In [ ]:
!yolo task=detect mode=predict model={MODEL}.pt source={HOME}/dog-2.jpeg save=True verbose=False

In [ ]:
from IPython.display import Image as IPyImage
IPyImage(filename=f'{HOME}/runs/detect/predict/dog-2.jpg', width=600)

### SDK + supervision

In [ ]:
from ultralytics import YOLO
from PIL import Image
import supervision as sv

model = YOLO(f'{MODEL}.pt')
image = Image.open(f'{HOME}/dog-2.jpeg')
result = model.predict(image, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)

def annotate(image, detections):
    ts = sv.calculate_optimal_text_scale(resolution_wh=image.size)
    box = sv.BoxAnnotator()
    lab = sv.LabelAnnotator(text_color=sv.Color.BLACK, text_scale=ts, smart_position=True)
    out = image.copy()
    out = box.annotate(out, detections)
    out = lab.annotate(out, detections)
    out.thumbnail((1000, 1000))
    return out

annotate(image, detections)

## Fine-tune YOLO26 trên dataset custom
Tải dataset 6-class (format `yolo26`). Slug Roboflow **viết thường** — thử lần lượt tên.

In [ ]:
!mkdir -p {HOME}/datasets
%cd {HOME}/datasets

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

project = version = dataset = None
for proj in ['datio_yolo', 'driver-yawn', 'driver-yawn-wh6wj']:
    try:
        project = rf.workspace('nguyen-tuan-dat').project(proj)
        version = project.version(1)
        dataset = version.download('yolo26')
        print('✅  Dùng project:', proj)
        break
    except Exception:
        print('  ⏭️ ', proj, 'không tải được, thử tiếp...')
%cd {HOME}

In [ ]:
# Vá data.yaml — bản export yolo26 của dataset này thiếu key train/val → xây lại từ thư mục
import yaml
from pathlib import Path
loc = Path(dataset.location)

old = {}
for yp in list(loc.glob('*.yaml')):
    with open(yp) as f: t = yaml.safe_load(f) or {}
    if 'names' in t: old = t; break
names = old.get('names')
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
if not names: names = ['close_eyeL','close_eyeR','no_yawn','open_eyeL','open_eyeR','yawn']

def imgdir(*c):
    for x in c:
        d = loc/x
        if d.exists() and any(d.iterdir()): return str(d.resolve())
    return None
cfg = {'train': imgdir('train/images','train'),
       'val':   imgdir('valid/images','valid','val') or imgdir('test/images','test'),
       'nc': len(names), 'names': names}
_t = imgdir('test/images','test')
if _t: cfg['test'] = _t
with open(f'{dataset.location}/data.yaml','w') as f: yaml.dump(cfg, f, sort_keys=False)
print('✅  data.yaml:', cfg)

## Custom Training

In [ ]:
%cd {HOME}
!yolo task=detect mode=train model={MODEL}.pt data={dataset.location}/data.yaml epochs=60 imgsz=640 batch=16 patience=20 plots=True

In [ ]:
import glob, os
TRAIN_DIR = max(glob.glob(f'{HOME}/runs/detect/train*'), key=os.path.getmtime)
BEST = f'{TRAIN_DIR}/weights/best.pt'
print('Train dir:', TRAIN_DIR)
!ls {TRAIN_DIR}

In [ ]:
from IPython.display import Image as IPyImage, display
for f in ['results.png','confusion_matrix.png','val_batch0_pred.jpg']:
    p = f'{TRAIN_DIR}/{f}'
    if os.path.exists(p): print(f); display(IPyImage(filename=p, width=620))

## Validate fine-tuned model

In [ ]:
!yolo task=detect mode=val model={BEST} data={dataset.location}/data.yaml

## Inference với model custom
### CLI

In [ ]:
!yolo task=detect mode=predict model={BEST} source={dataset.location}/test/images save=True verbose=False

### SDK + supervision (lưới 3×3 ngẫu nhiên)

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
import supervision as sv

best_model = YOLO(BEST)
ds_test = sv.DetectionDataset.from_yolo(
    images_directory_path=f'{dataset.location}/test/images',
    annotations_directory_path=f'{dataset.location}/test/labels',
    data_yaml_path=f'{dataset.location}/data.yaml')

color = sv.ColorPalette.from_hex(['#9999ff','#3399ff','#66ffff','#33ff99','#66ff66','#ff9b00'])
def annotate2(image, det):
    ts = sv.calculate_optimal_text_scale(resolution_wh=image.size)
    box = sv.BoxAnnotator(color=color)
    lab = sv.LabelAnnotator(color=color, text_color=sv.Color.BLACK, text_scale=ts, smart_position=True)
    out = image.copy(); out = box.annotate(out, det); out = lab.annotate(out, det); return out

N = min(9, len(ds_test))
imgs = []
for i in random.sample(range(len(ds_test)), N):
    path, _, _ = ds_test[i]
    im = Image.open(path)
    r = best_model.predict(im, verbose=False)[0]
    imgs.append(annotate2(im, sv.Detections.from_ultralytics(r)))

fig, axes = plt.subplots(3,3, figsize=(12,12))
for ax, im in zip(axes.flat, imgs): ax.imshow(im); ax.axis('off')
plt.subplots_adjust(wspace=0.02, hspace=0.02); plt.show()

## Lưu weights về Google Drive (dùng cho file combine / Android)

In [ ]:
from google.colab import drive
import shutil, json
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/DrowsyDriver_Results'); OUT.mkdir(parents=True, exist_ok=True)
shutil.copy(BEST, OUT/f'{MODEL}_best.pt')
(OUT/'summary_yolo26.json').write_text(json.dumps({'model':MODEL,'format':'yolo26','classes':names}, indent=2))
print('✅  Lưu Drive:', OUT/f'{MODEL}_best.pt')

## Deploy model lên Roboflow
Upload weights YOLO26 lên Roboflow Deploy để chạy inference trên hạ tầng Roboflow.

In [ ]:
try:
    project.version(version.version).deploy(model_type='yolo26', model_path=f'{TRAIN_DIR}/')
    print('✅  Đã deploy lên Roboflow')
except Exception as e:
    print('⏭️  Deploy skip:', str(e)[:120])

## Chạy inference từ model hosted trên Roboflow

In [ ]:
!pip install -q inference
try:
    from inference import get_model
    model_id = project.id.split('/')[1] + '/' + str(version.version)
    hosted = get_model(model_id, ROBOFLOW_API_KEY)
    imgs = []
    for i in random.sample(range(len(ds_test)), min(9, len(ds_test))):
        path, _, _ = ds_test[i]
        im = Image.open(path)
        r = hosted.infer(im)[0]
        imgs.append(annotate2(im, sv.Detections.from_inference(r)))
    fig, axes = plt.subplots(3,3, figsize=(12,12))
    for ax, im in zip(axes.flat, imgs): ax.imshow(im); ax.axis('off')
    plt.subplots_adjust(wspace=0.02, hspace=0.02); plt.show()
except Exception as e:
    print('⏭️  Hosted inference skip (cần deploy xong ở cell trên):', str(e)[:120])

## 🏆 Hoàn tất
- `best.pt` đã lưu Drive (`yolo26s_best.pt`)
- Đã deploy lên Roboflow + chạy thử model hosted
- Bước tiếp: chạy `colab_yolo11.ipynb` rồi `colab_combine_6class.ipynb` để so sánh + ensemble